# Golden Truth Generation — Skill Extraction

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

---

This notebook generates **golden truth skill labels** for the Coursera dataset  
using Google Gemini as a proxy annotator.

**Sections**
- Cell 1: Install dependencies
- Cell 2: Generate golden truth skills and export to CSV

**Disclosure note for methodology:**  
Golden truth skills are generated by Google Gemini (a model not used in any  
experiment being evaluated), serving as a silver-standard proxy for human annotation.

---

**Setup**
- Step 1: Run Cell 1 to install dependencies. Restart runtime if prompted.
- Step 2: Add your Gemini API key as a Colab Secret named `GEMINI_API_KEY`.
- Step 3: Run Cell 2.


In [ ]:
# =============================================================================
# CELL 1 — INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

PACKAGES = [
    ("google-generativeai>=0.5.0",),
    ("pandas>=2.0.0",),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


for pkg_group in PACKAGES:
    pip_install(*pkg_group)

print("All packages installed successfully.")


All packages installed successfully.


In [ ]:
# =============================================================================
# CELL 2 — SKILL EXTRACTION GOLDEN TRUTH GENERATION
#
# Sections
# --------
#   A.  Imports
#   B.  Configuration
#   C.  Dataset load
#   D.  API key setup  (supports up to 3 keys with automatic fallback)
#   E.  Prompt template
#   F.  Gemini caller
#   G.  Generation loop
#   H.  Quality checks
#   I.  Save & download
# =============================================================================


# =============================================================================
# A.  IMPORTS
# =============================================================================

import getpass
import html
import os
import random
import re
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import google.generativeai as genai
import pandas as pd
from google.colab import drive

try:
    from google.colab import files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

SEPARATOR_SM: int = 40
SEPARATOR_MD: int = 55
SEPARATOR_LG: int = 65


# =============================================================================
# B.  CONFIGURATION
# =============================================================================

# Gemini model to use as proxy annotator.
# Must NOT be any model used in the main experiment.
GEMINI_MODEL: str = "gemini-2.5-flash"

# Generation parameters
TEMPERATURE: float = 0.2   # Low temperature for consistent, factual output
TOP_P: float = 0.9
MAX_OUTPUT_TOKENS: int = 1024

# Inter-call delay to respect API rate limits (seconds)
DELAY_MIN: float = 8.0
DELAY_MAX: float = 12.0

# Retry settings for API calls
MAX_RETRIES: int = 3
RETRY_BACKOFF_MIN: float = 10.0
RETRY_BACKOFF_MAX: float = 20.0

# Max retries for skill generation if response is invalid
MAX_GENERATION_RETRIES: int = 2  # Including the initial attempt, so 2 retries means 3 total attempts

# Dataset
N_ROWS: int = 25
# Names of the three Colab Secrets — add each key under the matching name
GEMINI_SECRET_NAMES: List[str] = [
    "GEMINI_API_KEY",    # Key 1 (primary)
    "GEMINI_API_KEY_2",  # Key 2 (first fallback)
    "GEMINI_API_KEY_3",  # Key 3 (second fallback)
]

# Minimum number of skills expected in a valid response.
MIN_SKILL_COUNT: int = 2
# Maximum number of skills to extract per course.
MAX_SKILL_COUNT: int = 8


def print_config() -> None:
    """Print the current generation configuration."""
    total_calls = N_ROWS * (MAX_GENERATION_RETRIES + 1)  # Max possible calls
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Golden Truth Skill Extraction — Configuration")
    print("-" * SEPARATOR_SM)
    print(f"  Gemini model       : {GEMINI_MODEL}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  top_p              : {TOP_P}")
    print(f"  Max output tokens  : {MAX_OUTPUT_TOKENS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Max API retries    : {MAX_RETRIES}")
    print(f"  Max gen retries    : {MAX_GENERATION_RETRIES}")
    print(f"  Est. total calls   : {total_calls} (max possible)")
    print(f"  Est. time (max)    : ~{avg_delay_min:.1f} min")
    print(f"  Min skills count   : {MIN_SKILL_COUNT}")
    print(f"  Max skills count   : {MAX_SKILL_COUNT}")
    print(f"  API keys available : {len(GEMINI_SECRET_NAMES)}")


print_config()


# =============================================================================
# C.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load and clean the Coursera CSV dataset from Google Drive.

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns.

    Raises:
        FileNotFoundError: If the dataset CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)

    drive.mount("/content/drive")

    dataset_path = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    print(f"  Loaded {len(df)} courses successfully.")
    return df


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)


# =============================================================================
# D.  API KEY SETUP  (up to 3 keys, automatic quota-based fallback)
# =============================================================================

def load_api_key(secret_name: str, key_number: int) -> str:
    """Load a single Gemini API key from Colab Secrets or manual input.

    Args:
        secret_name: The Colab Secret key name (e.g. "GEMINI_API_KEY").
        key_number:  Human-readable key number (1, 2, or 3) for prompts.

    Returns:
        API key string, or empty string if the key was not provided.
    """
    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Key {key_number} loaded from Colab Secret '{secret_name}'.")
        except Exception:
            pass  # Key not set — will try manual entry for key 1 only

    if not api_key and key_number == 1:
        # Always prompt for the primary key if it is missing
        api_key = getpass.getpass(
            f"  Enter your primary Gemini API key ('{secret_name}'): "
        ).strip()

    if api_key:
        print(f"  Key {key_number} : AIza...{api_key[-4:]}")
    else:
        print(f"  Key {key_number} : not set (secret '{secret_name}' not found — skipped).")

    return api_key


def load_all_api_keys(secret_names: List[str]) -> List[str]:
    """Load all configured API keys, returning only non-empty ones.

    Args:
        secret_names: Ordered list of Colab Secret names to try.

    Returns:
        List of non-empty API key strings, in priority order.

    Raises:
        ValueError: If no valid key is found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    keys: List[str] = []
    for i, name in enumerate(secret_names, start=1):
        key = load_api_key(name, i)
        if key:
            keys.append(key)

    if not keys:
        raise ValueError(
            "No Gemini API key provided. "
            f"Add at least one key as a Colab Secret named '{secret_names[0]}' "
            "or enter it when prompted."
        )

    print(f"\n  {len(keys)} key(s) available for automatic fallback.")
    return keys


def build_gemini_client(api_key: str) -> genai.GenerativeModel:
    """Configure the genai library with the given key and return a client.

    Args:
        api_key: Gemini API key string.

    Returns:
        Configured GenerativeModel instance.
    """
    genai.configure(api_key=api_key)
    return genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=genai.GenerationConfig(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
    )


ALL_API_KEYS: List[str] = load_all_api_keys(GEMINI_SECRET_NAMES)

# Active key index; incremented on each quota error
_active_key_index: int = 0
GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
print(f"  Gemini client ready : {GEMINI_MODEL}  (using key 1 of {len(ALL_API_KEYS)})")


def rotate_api_key() -> bool:
    """Switch to the next available API key and rebuild the Gemini client.

    Returns:
        True if rotation succeeded, False if all keys are exhausted.
    """
    global _active_key_index, GEMINI_CLIENT

    next_index = _active_key_index + 1
    if next_index >= len(ALL_API_KEYS):
        return False  # No more keys

    _active_key_index = next_index
    GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
    print(
        f"\n  *** Rotated to API key {_active_key_index + 1} of {len(ALL_API_KEYS)} ***\n"
    )
    return True


# =============================================================================
# E.  PROMPT TEMPLATE
# =============================================================================

def build_skill_extraction_prompt(description: str) -> str:
    """Build the golden truth skill extraction prompt for a single course.

    The prompt instructs Gemini to extract concrete, teachable skills
    directly from the course description text only.

    Args:
        description: Cleaned course description string.

    Returns:
        Fully rendered prompt string.
    """
    return f"""You are an expert skills analyst for online learning content.

Task: Extract the key skills taught in this course based solely on the description below.

Requirements:
- Extract between {MIN_SKILL_COUNT} and {MAX_SKILL_COUNT} concrete, teachable skills
- Each skill must be a short noun phrase (1-4 words), e.g. "data visualisation", "regression modelling"
- Base extraction ONLY on the description, Do not infer from the course title.
- Return skills as a plain comma-separated list on a single line, no numbering, no bullets
- Do not include any preamble, explanation, or trailing punctuation

Description: {description}

Skills:"""


# =============================================================================
# F.  GEMINI CALLER
# =============================================================================

# Phrases that indicate a quota or rate-limit error
QUOTA_ERROR_PHRASES: Tuple[str, ...] = (
    "quota",
    "rate limit",
    "resource exhausted",
    "429",
)


class QuotaExceeded(Exception):
    """Raised when ALL available Gemini API keys have hit their quota.

    At this point no rotation is possible and the caller saves partial results.
    """


def is_quota_error(exc: Exception) -> bool:
    """Return True if the exception signals an API quota error.

    Args:
        exc: Exception raised by the Gemini client.

    Returns:
        True if any quota-related phrase is found in the error message.
    """
    return any(phrase in str(exc).lower() for phrase in QUOTA_ERROR_PHRASES)


def call_gemini(
    prompt: str,
    retries: int = MAX_RETRIES,
) -> Tuple[str, float]:
    """Call the Gemini API and return (response_text, latency_seconds).

    Applies a random inter-call delay before each attempt.
    On quota errors, automatically rotates to the next available API key
    and retries immediately. Raises QuotaExceeded only when all keys are
    exhausted.  Retries up to `retries` times with random back-off on
    transient (non-quota) errors.

    Args:
        prompt:  Fully rendered prompt string.
        retries: Number of retry attempts on transient errors.

    Returns:
        Tuple of (response_text, latency_in_seconds).

    Raises:
        QuotaExceeded: If all API keys have hit their quota limit.
    """
    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"    Waiting {delay:.1f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = GEMINI_CLIENT.generate_content(prompt)
            latency = round(time.perf_counter() - t_start, 3)
            text = response.text.strip() if response.text else ""
            print(f"OK ({latency}s)")
            return text, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)

            if "503" in str(exc) or "service unavailable" in str(exc).lower():
                print(f"\n    503 Service Unavailable (attempt {attempt}/{retries}). Retrying same key...")
                if attempt < retries:
                    backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                    print(f"    Retrying in {backoff:.1f}s...")
                    time.sleep(backoff)
                    continue
                else:
                    print("    All retries exhausted on 503. Recording empty response.")
                    return "", latency

            if is_quota_error(exc):
                print(f"\n    Quota limit hit on key {_active_key_index + 1}.")
                rotated = rotate_api_key()
                if not rotated:
                    raise QuotaExceeded(
                        f"All {len(ALL_API_KEYS)} API key(s) have hit their quota. "
                        "Original error: " + str(exc)
                    ) from exc
                print(f"    Retrying with key {_active_key_index + 1}...")
                delay2 = random.uniform(DELAY_MIN, DELAY_MAX)
                print(f"    Waiting {delay2:.1f}s...", end=" ", flush=True)
                time.sleep(delay2)
                attempt = 0
                continue

            print(f"\n    Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"    Retrying in {backoff:.1f}s...")
                time.sleep(backoff)
            else:
                print("    All retries exhausted. Recording empty response.")
                return "", latency


# =============================================================================
# G.  GENERATION LOOP
# =============================================================================

def parse_skills(raw_text: str) -> List[str]:
    """Parse a comma-separated skills string into a clean list.

    Args:
        raw_text: Raw model output string.

    Returns:
        List of stripped, non-empty skill strings.
    """
    if not raw_text:
        return []
    skills = [s.strip().strip('"\'.') for s in raw_text.split(',')]
    return [s for s in skills if s]


def build_gold_record(
    idx: int,
    row: pd.Series,
    skills: List[str],
    raw_response: str,
    latency: float,
    status: str,
) -> Dict:
    """Assemble a single gold skill record dict.

    Args:
        idx:          Row index in the dataset.
        row:          Dataset row as a Series.
        skills:       Parsed list of extracted skill strings.
        raw_response: Raw model output before parsing.
        latency:      API call latency in seconds.
        status:       "ok", "empty", or "too_few".

    Returns:
        Dictionary with all gold record fields.
    """
    return {
        "course_idx":          idx,
        "course_title":        str(row.get("course_title", "Unknown")),
        "course_organization": str(row.get("course_organization", "N/A")),
        "course_difficulty":   str(row.get("course_difficulty", "N/A")),
        "description":         str(row.get("description", "")),
        "gold_skills":         ", ".join(skills),
        "gold_skills_list":    skills,
        "skill_count":         len(skills),
        "raw_response":        raw_response,
        "latency_seconds":     latency,
        "status":              status,
        "gemini_model":        GEMINI_MODEL,
        "api_key_index":       _active_key_index + 1,
        "temperature":         TEMPERATURE,
        "timestamp":           datetime.now(timezone.utc).isoformat(),
    }


gold_records: List[Dict] = []
quota_hit: bool = False
generation_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("GENERATING GOLDEN TRUTH SKILL LABELS")
print("=" * SEPARATOR_LG)

try:
    for idx, row in DATASET_DF.iterrows():
        title = str(row.get("course_title", "Unknown"))
        description = str(row.get("description", ""))

        print(f"  [{idx:02d}] {title[:60]}...")

        prompt = build_skill_extraction_prompt(description)

        current_skills: List[str] = []
        current_raw: str = ""
        current_latency: float = 0.0
        current_status: str = "failed_too_few"

        for attempt in range(MAX_GENERATION_RETRIES + 1):
            if attempt > 0:
                print(f"    Retrying generation (attempt {attempt + 1}/{MAX_GENERATION_RETRIES + 1}) for 'too_few'...")
                time.sleep(random.uniform(2.0, 5.0))

            raw_response, latency = call_gemini(prompt)
            skills = parse_skills(raw_response)

            if not raw_response:
                status = "empty"
            elif len(skills) < MIN_SKILL_COUNT:
                status = "too_few"
            else:
                status = "ok"

            current_skills = skills
            current_raw = raw_response
            current_latency = latency
            current_status = status

            if status == "ok":
                break

        gold_records.append(
            build_gold_record(idx, row, current_skills, current_raw, current_latency, current_status)
        )

except QuotaExceeded as exc:
    quota_hit = True
    print("\n" + "!" * SEPARATOR_LG)
    print("*** ALL API KEYS QUOTA EXCEEDED — generation stopped here ***")
    print(f"  Records collected : {len(gold_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)

GOLD_DF: pd.DataFrame = pd.DataFrame(gold_records)
generation_status = "STOPPED EARLY (all keys quota exhausted)" if quota_hit else "COMPLETE"
print(f"\nGeneration {generation_status}. Total records: {len(GOLD_DF)}")


# =============================================================================
# H.  QUALITY CHECKS
# =============================================================================

def run_quality_checks(gold_df: pd.DataFrame) -> None:
    """Print a quality report on the generated gold skill labels.

    Checks performed:
    - Status distribution (ok / empty / too_few)
    - Skill count statistics
    - Top-frequency skills across all courses
    - Latency summary

    Args:
        gold_df: DataFrame produced by the generation loop.
    """
    if gold_df.empty:
        print("No records to check.")
        return

    print("\n" + "=" * SEPARATOR_LG)
    print("QUALITY REPORT")
    print("=" * SEPARATOR_LG)

    # Status distribution
    print("\n  Status distribution")
    print("  " + "-" * SEPARATOR_SM)
    for status, count in gold_df["status"].value_counts().items():
        pct = count / len(gold_df) * 100
        print(f"    {status:<15} {count:>3}  ({pct:.1f}%)")

    # API key usage breakdown
    if "api_key_index" in gold_df.columns:
        print("\n  API key usage")
        print("  " + "-" * SEPARATOR_SM)
        for key_idx, count in gold_df["api_key_index"].value_counts().sort_index().items():
            print(f"    Key {key_idx} : {count} record(s)")

    # Skill count statistics
    ok_df = gold_df[gold_df["status"] == "ok"]
    if not ok_df.empty:
        sc = ok_df["skill_count"]
        print("\n  Skill count per course (status=ok)")
        print("  " + "-" * SEPARATOR_SM)
        print(f"    Min    : {sc.min()}")
        print(f"    Max    : {sc.max()}")
        print(f"    Mean   : {sc.mean():.1f}")
        print(f"    Median : {sc.median():.1f}")

    # Top skills frequency
    from collections import Counter
    all_skills: List[str] = []
    for skills_list in gold_df["gold_skills_list"]:
        if isinstance(skills_list, list):
            all_skills.extend([s.lower() for s in skills_list])
    if all_skills:
        top_skills = Counter(all_skills).most_common(10)
        print("\n  Top 10 most frequent skills")
        print("  " + "-" * SEPARATOR_SM)
        for skill, cnt in top_skills:
            print(f"    {skill:<40} {cnt}")

    # Latency summary
    lat = gold_df["latency_seconds"]
    print("\n  Latency (seconds)")
    print("  " + "-" * SEPARATOR_SM)
    print(f"    Min    : {lat.min():.3f}s")
    print(f"    Max    : {lat.max():.3f}s")
    print(f"    Mean   : {lat.mean():.3f}s")
    print(f"    Total  : {lat.sum():.1f}s")

    print("\n" + "=" * SEPARATOR_LG)


run_quality_checks(GOLD_DF)


# =============================================================================
# I.  SAVE & DOWNLOAD
# =============================================================================

GOLD_FILENAME: str = f"gold_skills_coursera.csv"
FLAGGED_FILENAME: str = f"gold_skills_flagged_coursera.csv"


def save_gold_skills(
    gold_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save the gold skill DataFrame to CSV and optionally trigger downloads.

    Saves two files:
    - Full output: all records including failures.
    - Flagged output: only records with status != 'ok', for manual review.

    Args:
        gold_df:  DataFrame produced by the generation loop.
        in_colab: True if running inside Google Colab.
    """
    # Drop list column before CSV export (lists don't serialise cleanly)
    export_df = gold_df.drop(columns=["gold_skills_list"], errors="ignore")
    export_df.to_csv(GOLD_FILENAME, index=False)
    print(f"\nSaved: {GOLD_FILENAME}  ({len(export_df)} rows)")

    flagged_df = export_df[export_df["status"] != "ok"]
    if not flagged_df.empty:
        flagged_df.to_csv(FLAGGED_FILENAME, index=False)
        print(
            f"Saved: {FLAGGED_FILENAME}  ({len(flagged_df)} rows — manual review needed)"
        )
    else:
        print("No flagged records — all skill extractions passed status checks.")

    if in_colab:
        print("\nDownloading files...")
        files.download(GOLD_FILENAME)
        if not flagged_df.empty:
            files.download(FLAGGED_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")

save_gold_skills(GOLD_DF, IN_COLAB)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Golden Truth Skill Extraction — Configuration
----------------------------------------
  Gemini model       : gemini-2.5-flash
  Temperature        : 0.2
  top_p              : 0.9
  Max output tokens  : 1024
  Courses            : 25
  Max API retries    : 3
  Max gen retries    : 2
  Est. total calls   : 75 (max possible)
  Est. time (max)    : ~12.5 min
  Min skills count   : 2
  Max skills count   : 8
  API keys available : 3

----------------------------------------
Dataset Load
----------------------------------------
Mounted at /content/drive
  Found dataset at: /content/drive/MyDrive/research/data/coursera_clean_sample.csv
  Loaded 25 courses successfully.

----------------------------------------
API Key Setup
----------------------------------------
  Key 1 loaded from Colab Secret 'GEMINI_API_KEY'.
  Key 1 : AIza...B9O0
  Key 2 loaded from Colab Secret 'GEMINI_API_KEY_2'.
  Key 2 : AIza...Oa8U
  Key 3 loaded from Colab Secret 'GEMINI_API_KEY_3'.
  Key 3 : AIza...Frdo

  3 ke

OK (1.608s)
  [22] elastic google cloud infrastructure scaling and automation...
    Waiting 10.9s... OK (5.002s)
  [23] genai for devops practitioners...
    Waiting 9.9s... OK (5.69s)
  [24] the science of the solar system...
    Waiting 11.8s... OK (4.547s)

Generation COMPLETE. Total records: 25

QUALITY REPORT

  Status distribution
  ----------------------------------------
    ok               25  (100.0%)

  API key usage
  ----------------------------------------
    Key 1 : 21 record(s)
    Key 2 : 4 record(s)

  Skill count per course (status=ok)
  ----------------------------------------
    Min    : 4
    Max    : 11
    Mean   : 7.2
    Median : 8.0

  Top 10 most frequent skills
  ----------------------------------------
    critical thinking                        2
    descriptive statistics                   2
    cloud security                           1
    classic security techniques              1
    web service security                     1
    cloud security 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
